<a href="https://colab.research.google.com/github/AppDevIQ/datascreeniq-python/blob/main/examples/integrations/colab/datascreeniq_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ DataScreenIQ — Interactive Demo

**Screen any data payload and get PASS / WARN / BLOCK in milliseconds.**

This notebook lets you try DataScreenIQ in 60 seconds:
1. Install the SDK
2. Screen sample data with DemoClient (no API key needed)
3. See the quality report
4. Try with your own data

**DemoClient** runs locally — checks null rates, type mismatches, and empty strings. No network calls, no signup.

For full screening (18 checks, drift detection, schema fingerprinting), get a free API key: [datascreeniq.com](https://datascreeniq.com)

---

## 1. Install

In [1]:
!pip install datascreeniq[pandas] --upgrade -q
import datascreeniq as dsiq
print(f"DataScreenIQ SDK v{dsiq.__version__} installed ✓")

DataScreenIQ SDK v1.0.12 installed ✓


## 2. Set up your client

**DemoClient** works instantly — no API key, no signup, runs locally.

Want full screening? Uncomment Option A and paste your key.

In [2]:
# Option A: Use your real API key (full 18 checks + drift detection)
# client = dsiq.Client("dsiq_live_YOUR_KEY_HERE")

# Option B: Demo client (no key needed, runs locally)
client = dsiq.DemoClient()
print("Client ready ✓")
print("Mode: Demo (null rates, type mismatches, empty strings)")
print("For full screening: get a free key at datascreeniq.com")

Client ready ✓
Mode: Demo (null rates, type mismatches, empty strings)
For full screening: get a free key at datascreeniq.com


## 3. Screen bad data — see a BLOCK

In [3]:
# This data has problems: type mismatch in 'amount', nulls in 'email'
bad_data = [
    {"order_id": "ORD-001", "amount": 99.50,    "email": "alice@corp.com", "status": "paid"},
    {"order_id": "ORD-002", "amount": "broken", "email": None,             "status": "paid"},
    {"order_id": "ORD-003", "amount": 75.00,    "email": None,             "status": "pending"},
]

report = client.screen(bad_data, source="orders")

print(report.summary())
print(f"\nStatus:          {report.status}")
print(f"Health:          {report.health_pct}")
print(f"Type mismatches: {report.type_mismatches}")
print(f"Null rates:      {report.null_rates}")
print(f"Latency:         {report.latency_ms}ms")

🚨 BLOCK | Health: 66.7% | Rows: 3 | Type mismatches: amount | Null rate: email=67% | (0ms)

Status:          BLOCK
Health:          66.7%
Type mismatches: ['amount']
Null rates:      {'email': 0.67}
Latency:         0ms


## 4. Screen clean data — see a PASS

In [4]:
clean_data = [
    {"order_id": "ORD-001", "amount": 99.50,  "email": "alice@corp.com", "status": "paid"},
    {"order_id": "ORD-002", "amount": 150.00, "email": "bob@corp.com",   "status": "paid"},
    {"order_id": "ORD-003", "amount": 75.00,  "email": "carol@corp.com", "status": "pending"},
    {"order_id": "ORD-004", "amount": 220.50, "email": "dave@corp.com",  "status": "paid"},
]

report = client.screen(clean_data, source="orders")
print(report.summary())

✅ PASS | Health: 100.0% | Rows: 4 | (0ms)


## 5. Screen a pandas DataFrame

In [5]:
import pandas as pd

df = pd.DataFrame([
    {"user_id": 1, "name": "Alice",   "age": 28,      "signup": "2025-01-15"},
    {"user_id": 2, "name": "Bob",     "age": "thirty", "signup": "2025-02-20"},
    {"user_id": 3, "name": None,      "age": 35,      "signup": "2025-03-10"},
    {"user_id": 4, "name": "Diana",   "age": 42,      "signup": None},
    {"user_id": 5, "name": "",        "age": 29,      "signup": "2025-05-01"},
])

print("Input DataFrame:")
display(df)

report = client.screen_dataframe(df, source="users")
print(f"\n{report.summary()}")

Input DataFrame:


,user_id,name,age,signup
0,1,Alice,28,2025-01-15
1,2,Bob,thirty,2025-02-20
2,3,None,35,2025-03-10
3,4,Diana,42,None
4,5,,29,2025-05-01



⚠️ WARN | Health: 92.0% | Rows: 5 | Type mismatches: age | (0ms)


## 6. Use as a pipeline guard

In [6]:
from datascreeniq.exceptions import DataQualityError

try:
    # This will raise DataQualityError because the data is bad
    client.screen(bad_data, source="orders").raise_on_block()
    print("✅ Data is clean — safe to load to warehouse")

except DataQualityError as e:
    print(f"🚨 Pipeline blocked: {e}")
    print(f"   Issues: {list(e.report.issues.keys())}")
    print(f"   → Route to dead-letter queue or alert team")

🚨 Pipeline blocked: Data blocked: health=66.7%, issues=['type_mismatches', 'null_rates']
   Issues: ['type_mismatches', 'null_rates']
   → Route to dead-letter queue or alert team


## 7. Explore the full JSON report

In [7]:
import json

report = client.screen(bad_data, source="orders")

print(json.dumps(report.to_dict(), indent=2))

{
  "request_id": "demo_01887a932b",
  "status": "BLOCK",
  "health_score": 0.6667,
  "decision": {
    "action": "BLOCK",
    "reason": "Type mismatch in: 'amount'; High null rate in 'email' (67%)"
  },
  "schema": {
    "order_id": {
      "type": "string",
      "confidence": 1.0
    },
    "amount": {
      "type": "number",
      "confidence": 0.67
    },
    "email": {
      "type": "string",
      "confidence": 1.0
    },
    "status": {
      "type": "string",
      "confidence": 1.0
    }
  },
  "schema_fingerprint": "ad3ac03635aa",
  "issues": {
    "type_mismatches": {
      "amount": {
        "expected": "number",
        "found": [
          "string"
        ],
        "sample_value": "broken",
        "rate": 0.33,
        "severity": "critical"
      }
    },
    "null_rates": {
      "email": {
        "actual": 0.67,
        "threshold": 0.3,
        "severity": "warning"
      }
    }
  },
  "drift": [],
  "stats": {
    "rows_received": 3,
    "rows_sampled": 3,
   

## 8. Upload your own CSV

Click the folder icon on the left → Upload a CSV file, then run:

In [8]:
# Uncomment and update the path to your CSV:
# report = client.screen_file("/content/your_data.csv", source="my-data")
# print(report.summary())

---

## What's different with a real API key?

| Feature | DemoClient | Client (API key) |
|---------|-----------|------------------|
| Null rate detection | ✅ | ✅ |
| Type mismatch detection | ✅ | ✅ |
| Empty string detection | ✅ | ✅ |
| Schema drift detection | ❌ | ✅ |
| IQR outlier detection | ❌ | ✅ |
| HyperLogLog distinct counts | ❌ | ✅ |
| Enum/cardinality tracking | ❌ | ✅ |
| Timestamp recency checks | ❌ | ✅ |
| Baseline adaptation (EMA) | ❌ | ✅ |
| Configurable thresholds | ❌ | ✅ |
| Dashboard & job history | ❌ | ✅ |

Get a free API key (500K rows/month): [datascreeniq.com](https://datascreeniq.com)

---

## Links

- [Python SDK (PyPI)](https://pypi.org/project/datascreeniq/)
- [GitHub](https://github.com/AppDevIQ/datascreeniq-python)
- [API reference](https://datascreeniq.com/api-reference.html)
- [Documentation](https://datascreeniq.com/docs)

Questions? → [app@datascreeniq.com](mailto:app@datascreeniq.com)